# Visualisation des données du Dataset Plant Village
Charlotte Chanudet et Mahaut 

Ce notebook sert de visualisation des données du dataset utilisé dans le cadre du projet d'Atelier I en IA pour les cours à l'UQAC. 
Il présente différents graphiques qui seront tous accompagnés d'une courte analyse et d'une explication sur pourquoi nous avons voulu faire ce graphique. 

In [2]:
# Cellule de préparation des données pour tous les graphiques
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Chemin du dataset
DATASET_PATH = r"dataset/Plant_leave_diseases_dataset_without_augmentation"

# --- Comptage des images par classe ---
folders = [f for f in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, f))]
data = []
for folder in folders:
    folder_path = os.path.join(DATASET_PATH, folder)
    nb_images = len([file for file in os.listdir(folder_path) if file.lower().endswith((".jpg", ".jpeg", ".png"))])
    data.append({"Classe": folder, "Nb_images": nb_images})
df = pd.DataFrame(data)
df = df.sort_values("Nb_images", ascending=False)

# --- Ajout colonne Etat (Sain/Malade) ---
df["Etat"] = df["Classe"].apply(lambda x: "Saine" if "healthy" in x.lower() else "Malade")
etat_count = df.groupby("Etat")["Nb_images"].sum()

# --- Séparation Plante/Etat pour histogramme groupé ---
data_split = []
for folder in folders:
    if "___" not in folder:
        continue
    plante, etat = folder.split("___", 1)
    folder_path = os.path.join(DATASET_PATH, folder)
    nb_images = len([file for file in os.listdir(folder_path) if file.lower().endswith((".jpg", ".jpeg", ".png"))])
    data_split.append({"Plante": plante, "Etat": etat, "Nb_images": nb_images})
df_split = pd.DataFrame(data_split)
malades = df_split[~df_split["Etat"].str.lower().str.contains("healthy")].groupby("Plante")["Nb_images"].sum()
saines = df_split[df_split["Etat"].str.lower().str.contains("healthy")].groupby("Plante")["Nb_images"].sum()
resultat = pd.concat([malades, saines], axis=1)
resultat.columns = ["Malade", "Saine"]
resultat = resultat.fillna(0).sort_values(by="Malade", ascending=False)

# --- Dimensions des images ---
widths, heights = [], []
for folder in folders:
    if "___" not in folder:
        continue
    folder_path = os.path.join(DATASET_PATH, folder)
    for file in os.listdir(folder_path):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            try:
                img_path = os.path.join(folder_path, file)
                img = Image.open(img_path)
                w, h = img.size
                widths.append(w)
                heights.append(h)
            except:
                continue
ratios = [w/h for w, h in zip(widths, heights) if h != 0]

# --- Couleurs moyennes (healthy/malade) ---
def get_mean_rgb(image_path):
    img = Image.open(image_path).resize((64, 64))
    img = np.array(img)
    if len(img.shape) == 3:
        return img.mean(axis=(0,1))
    else:
        return None
healthy_colors, diseased_colors = [], []
for folder in folders:
    if "___" not in folder:
        continue
    plante, etat = folder.split("___", 1)
    folder_path = os.path.join(DATASET_PATH, folder)
    for file in os.listdir(folder_path):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            path = os.path.join(folder_path, file)
            rgb = get_mean_rgb(path)
            if rgb is None or not isinstance(rgb, np.ndarray) or rgb.shape != (3,):
                continue
            if "healthy" in etat.lower():
                healthy_colors.append(rgb)
            else:
                diseased_colors.append(rgb)
healthy_colors = np.array(healthy_colors)
diseased_colors = np.array(diseased_colors)


KeyboardInterrupt: 

In [ ]:
# Histogramme 1 : répartition par dossier
plt.figure(figsize=(15,6))
plt.bar(df["Classe"], df["Nb_images"])
plt.xticks(rotation=90)
plt.xlabel("Classes")
plt.ylabel("Nombre d'images")
plt.title("Répartition des images par classe")
plt.tight_layout()
plt.show()

Dans un premier temps nous avons affiché la répartition par dossiers, afin de savoir globalement combien nous 
avons d'images globalement et de savoir si certaines espèces sont sous représentées par rapport à d'autres. Cette information nous permettra de savoir sur quels dossiers nous allons devoir faire des augmentations pour ne pas encore plus représenter des espèces qui sont déjà très présentes

On se rend effectivement compte que certains dossiers comme les tomates ou oranges comportent presque 5000 images alors que d'autres comme les pommes de terre saines en ont beaucoup moins. 

In [ ]:
# Histogramme 2 : sain vs malade
plt.figure(figsize=(6,5))
plt.bar(etat_count.index, etat_count.values)
plt.xlabel("État")
plt.ylabel("Nombre d'images")
plt.title("Répartition des plantes saines et malades")
for i, v in enumerate(etat_count.values):
    plt.text(i, v, str(v), ha='center')
plt.show()

Pour l'histogramme des plantes saines ou malades, nous voulions voir quelle proportion d'images étaient de 
plantes malades, afin de nous faire une idée tout simplement.

Cet histogramme montre 40 000 images de plantes malades contre 15 000 de plantes saines, ce qui fait qu'un peu de 70% du dataset est composé de plantes malades. 

In [ ]:
# Histogramme groupé : Malades vs Saines par plante
import numpy as np
x = np.arange(len(resultat))
largeur = 0.4
plt.figure(figsize=(16,7))
bar1 = plt.bar(x-largeur/2, resultat["Malade"], largeur, label="Malade")
bar2 = plt.bar(x+largeur/2, resultat["Saine"], largeur, label="Saine")
plt.xticks(x, resultat.index, rotation=45)
plt.xlabel("Plantes")
plt.ylabel("Nombre d'images")
plt.title("Comparaison images : Malades vs Saines")
plt.legend()
for bar in bar1:
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height(), int(bar.get_height()), ha='center', fontsize=8)
for bar in bar2:
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height(), int(bar.get_height()), ha='center', fontsize=8)
plt.tight_layout()
plt.show()

La visualisation des plantes malades et saines par type de plante nous permets de voir si certaines plantes ne 
sont représentées que saines ou que malades. Ce qui pourrait poser des soucis si elles ne sont que malades car la plante n'apprend pas à quoi ressemble la plante saine.

C'est notamment le cas pour l'orange et le squash, qui n'ont que des représentations malades, et à l'inverse pour les framboises, soya et bleuets, nous n'avons que des représentations saines. Les représentations saines posent moins de soucis car elles donnent des nouvelles plantes et type de feuilles à analyser. Cependant pour les oranges et les squash c'est un peu plus problématique car le modèle n'aura aucune idée d'à quoi ressemblent une feuille saine de ces plantes. Il pourra détecter la maladie mais risque de détecter une feuille saine comme d'une autre plante ou malade. 

In [ ]:
# Nuage de points des dimensions des images
plt.figure(figsize=(8,6))
plt.scatter(widths, heights, alpha=0.3)
plt.xlabel("Largeur (pixels)")
plt.ylabel("Hauteur (pixels)")
plt.title("Distribution des dimensions des images")
plt.show()

Ce nuage nous sert à vérifier si toutes les images ont la même taille ou si certaines doivent être reformatées.

En l'occurence elles font toutes la même taille ce qui règle le problème. 

In [ ]:
# Histogramme des ratios d'aspect
plt.figure(figsize=(8,5))
plt.hist(ratios, bins=50)
plt.xlabel("Aspect Ratio (largeur / hauteur)")
plt.ylabel("Nombre d'images")
plt.title("Distribution des ratios d'aspect des images")
plt.show()

Ici nous voulions voir si la forme des images, elles sont toutes des carrés, puisque le ratio largeur/hauteur
est de 1.

In [ ]:
# 3. Distribution des intensités de couleur (R, G, B) pour chaque état
import matplotlib.pyplot as plt
plt.figure(figsize=(12,5))
for i, color in enumerate(["Rouge", "Vert", "Bleu"]):
    plt.subplot(1,3,i+1)
    plt.hist(healthy_colors[:,i], bins=40, alpha=0.6, label="Sain", color="g")
    plt.hist(diseased_colors[:,i], bins=40, alpha=0.6, label="Malade", color="r")
    plt.title(f"Distribution {color}")
    plt.xlabel("Intensité")
    plt.ylabel("Nombre d'images")
    if i==0:
        plt.legend()
plt.tight_layout()
plt.show()

Ces trois graphiques nous permettent de nous faire une idée sur si les plantes malades ont des intensités de 
couleurs différentes ou si globalement la répartition est la même.

In [ ]:
# 2. Heatmap croisant plantes et états
import seaborn as sns
import matplotlib.pyplot as plt
pivot = df_split.pivot_table(index="Plante", columns="Etat", values="Nb_images", fill_value=0)
plt.figure(figsize=(14,8))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlGnBu")
plt.title("Heatmap du nombre d'images par plante et état")
plt.ylabel("Plante")
plt.xlabel("État")
plt.show()

Enfin cette heatmap nous permet une visualisation des différents types de maladies en fonction des plantes avec 
le nombre d'images par état.